# Week 6 

Solution: Taking the 90th percentile of the treatment effect and use it ass a proxy for the Marginal Treatment Effect. 

In order to migitate the optimization bias and the impact of estimated noise, we can use a stasticcal measure like the 90th percentile. Using the 90th percentile of the estimated treatment effects among the untreated, we trim off the extreme outliers that are most likely heavily influenced by random error. This will provide a more stable and realistic upper-bound estimate of what the optimal treatment effect could actually be. This captures the subset of the population that benefits the most without being affected by loud noise points. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# 1. Load the dataset (using our prior homework dataset as an example)
df = pd.read_csv('homework_6.1.csv')

# Separate treated (X=1) and untreated (X=0) items
treated = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# 2. Fit NearestNeighbors models based on the confounder Z
nn_treated = NearestNeighbors(n_neighbors=1).fit(treated[['Z']])

# 3. Find counterfactuals for the untreated items
# For untreated items: closest treated item corresponds to their counterfactual outcome
_, dist_untreated = nn_treated.kneighbors(untreated[['Z']])
te_untreated = treated.iloc[dist_untreated.flatten()]['Y'].values - untreated['Y']

# 4. Calculate Maximum vs. 90th Percentile Treatment Effect
# The problem: taking the exact max exposes us to extreme noise/bias
naive_max_mte = te_untreated.max()

# The solution: take the 90th percentile as a robust proxy
robust_90th_mte = np.percentile(te_untreated, 90)

print(f"Naive Maximum Treatment Effect  : {naive_max_mte:.4f}")
print(f"Robust 90th Percentile Estimate : {robust_90th_mte:.4f}")

Naive Maximum Treatment Effect  : 2.1725
Robust 90th Percentile Estimate : 1.9280


# Week 7

In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Generate data with a hidden confounder (Z)
np.random.seed(42)
n = 1000
Z = np.random.normal(0, 1, n)          # Confounder
X = 1.0 * Z + np.random.normal(0, 1, n)  # X is correlated with Z
# Y is caused by X (effect = 1.0) and Z (effect = 2.0)
Y = 1.0 * X + 2.0 * Z + np.random.normal(0, 1, n) 

df = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})

# 2. Regress Y on X and Z (Correct Causal Model)
model_correct = sm.OLS(df['Y'], sm.add_constant(df[['X', 'Z']])).fit()

# 3. Regress Y on X only (Biased Omitted Variable Model)
model_biased = sm.OLS(df['Y'], sm.add_constant(df['X'])).fit()

print(f"True Effect (beta_X) used in simulation: 1.0")
print(f"Estimated coefficient (Correct Model): {model_correct.params['X']:.4f}")
print(f"Estimated coefficient (Biased Model):  {model_biased.params['X']:.4f}")

if model_biased.params['X'] > model_correct.params['X']:
    print("\nResult: The correlation is OVERESTIMATED.")
elif model_biased.params['X'] < model_correct.params['X']:
    print("\nResult: The correlation is UNDERESTIMATED.")


True Effect (beta_X) used in simulation: 1.0
Estimated coefficient (Correct Model): 0.9898
Estimated coefficient (Biased Model):  1.9863

Result: The correlation is OVERESTIMATED.


In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Setup for the simulation
np.random.seed(42)
n = 100
n_trials = 1000
p_values_w = []

for i in range(n_trials):
    # W and X are pure noise (no relationship to each other)
    W = np.random.normal(0, 1, n)
    X = np.random.normal(0, 1, n)
    # Y depends ONLY on X (coefficient of W is zero)
    Y = 2 * X + np.random.normal(0, 1, n)
    
    # Fit regression model
    df_sim = pd.DataFrame({'W': W, 'X': X, 'Y': Y})
    model = sm.OLS(df_sim['Y'], sm.add_constant(df_sim[['W', 'X']])).fit()
    
    # Store p-value for W
    p_values_w.append(model.pvalues['W'])

# 2. Report results
min_p = min(p_values_w)
count_below_05 = sum(p < 0.05 for p in p_values_w)

print(f"Number of trials: {n_trials}")
print(f"Minimum p-value found: {min_p:.6f}")
print(f"Number of trials where p < 0.05: {count_below_05}")
print(f"Percentage of Type I errors: {count_below_05 / n_trials * 100:.1f}%")


Number of trials: 1000
Minimum p-value found: 0.000475
Number of trials where p < 0.05: 51
Percentage of Type I errors: 5.1%


# Week 8

When implementing IPW to identify the ATE, I ensured that every subject had a non-zero probability of being in either the treatment or control group. Using the LogisticRegression() model to estimate propensity scores by using only a single linear term for z also lead to another insight. If the true relationship between the covariate and the treatment is non-linear, the model will produce biased propensity scores. There is also the assumption that all variables affecting both the treatment and outcome are observed and included. Since this dataset only provides one covariate, if there are other unobserved variables influencing the treatment and outcome the identified ATE will be biased. 

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

# 1. Load the data
df_81 = pd.read_csv("homework_8.1.csv")

# 2. Estimate propensity scores using logistic regression
# Z values predict X (treatment)
Z = df_81[['Z']]
X = df_81['X']

# Using sklearn for propensity score estimation
propensity_model = LogisticRegression()
propensity_model.fit(Z, X)

# Calculate P(X=1 | Z)
# predict_proba returns [P(X=0), P(X=1)]
df_81['propensity_score'] = propensity_model.predict_proba(Z)[:, 1]

# 3. Calculate inverse probability weights
# Weight = 1/P for X=1, and 1/(1-P) for X=0
df_81['ip_weight'] = np.where(df_81['X'] == 1, 
                              1 / df_81['propensity_score'], 
                              1 / (1 - df_81['propensity_score']))

# 4. Estimate the ATE
# ATE = Mean(Y * weight | X=1) - Mean(Y * weight | X=0)
# But standard ATE via IPW is (1/N) * sum( X*Y/P - (1-X)*Y/(1-P) )
# or more simply, the difference of weighted means
weighted_y_treat = (df_81[df_81['X'] == 1]['Y'] * df_81[df_81['X'] == 1]['ip_weight']).sum() / len(df_81)
weighted_y_control = (df_81[df_81['X'] == 0]['Y'] * df_81[df_81['X'] == 0]['ip_weight']).sum() / len(df_81)

ate_ipw = weighted_y_treat - weighted_y_control

print(f"Propensity Score Head:\n{df_81[['X', 'Z', 'propensity_score', 'ip_weight']].head()}\n")
print(f"Estimated ATE using IPW: {ate_ipw:.4f}")

Propensity Score Head:
   X         Z  propensity_score  ip_weight
0  1  1.764052          0.840114   1.190315
1  0  0.400157          0.584646   2.407585
2  0  0.978738          0.711082   3.461195
3  0  2.240893          0.892793   9.327719
4  1  1.867558          0.853089   1.172211

Estimated ATE using IPW: 2.2393


C:\Users\santo\AppData\Local\Temp\ipykernel_12816\2693009216.py:20: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_81['propensity_score'] = propensity_model.predict_proba(Z)[:, 1]
C:\Users\santo\AppData\Local\Temp\ipykernel_12816\269300921

In implementing the Mahalanobis distance there were a few technical and conceptual obstacles. Since it uses a nested structure, the operation is performed in a non-vectorized manner causing it to take a longer amount of time. In matching there is an assumption that for every treated unit there is a comparable control unit, however even if a nearest neighbor is found, the Mahalanobis distance is large and the counterfactual may not actually be similar. This can lead to matching bias.  

In [3]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import mahalanobis

# 1. Load the dataset
df_82 = pd.read_csv("homework_8.2.csv")

# 2. Prepare the inverse covariance matrix
# Use all Z1 and Z2 values, make them into a 2xN matrix (Z1 and Z2 as rows)
# In pandas, this is Z1 and Z2 columns. df[['Z1', 'Z2']].values.T gives 2xN
z_matrix = df_82[['Z1', 'Z2']].values.T
# Find the 2x2 covariance and invert
cov_matrix = np.cov(z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

# 3. Separate treated and untreated
treated = df_82[df_82['X'] == 1].copy()
untreated = df_82[df_82['X'] == 0].copy()

def find_nearest_mahalanobis(t_row, u_pool, i_cov):
    z_t = t_row[['Z1', 'Z2']].values
    distances = u_pool.apply(lambda u_row: mahalanobis(z_t, u_row[['Z1', 'Z2']].values, i_cov), axis=1)
    # Match to the single nearest untreated item (index of minimum distance)
    return distances.idxmin()

# 4. Perform matching with replacement
# For each treated item, find the index of the nearest untreated neighbor
treated['match_idx'] = treated.apply(lambda r: find_nearest_mahalanobis(r, untreated, inv_cov), axis=1)

# 5. Calculate results (e.g., ATT)
treated['Y_counterfactual'] = untreated.loc[treated['match_idx'], 'Y'].values
treated['treatment_effect'] = treated['Y'] - treated['Y_counterfactual']

att_mahalanobis = treated['treatment_effect'].mean()

print(f"Inverse Covariance Matrix:\n{inv_cov}\n")
print(f"First 5 Matches:\n{treated[['X', 'Y', 'match_idx', 'Y_counterfactual']].head()}\n")
print(f"Average Treatment Effect on the Treated (ATT): {att_mahalanobis:.4f}")

Inverse Covariance Matrix:
[[ 2.02734407 -1.03387633]
 [-1.03387633  1.06684813]]

First 5 Matches:
   X         Y  match_idx  Y_counterfactual
0  1  4.652085        681          0.428954
1  1  2.590221        752         -0.034844
2  1  3.898981        892          1.164988
3  1  5.857179        922          1.797450
4  1  3.647489        922          1.797450

Average Treatment Effect on the Treated (ATT): 3.4377


C:\Users\santo\AppData\Local\Temp\ipykernel_12816\285755188.py:31: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  treated['Y_counterfactual'] = untreated.loc[treated['match_idx'], 'Y'].values
C:\Users\santo\AppData\Local\Temp\ipykernel_12816\

Implementing the vectorized distance calculation to identify specific outliers is intended to find where the common support is the weakest. In this situation, the high distance means the treated item is in a lonely part of the feature space where no control units exist. Another challenge was that all_dists(outlier_idx) returns the distance vector for the i-th row of the treated dataframe. By mapping the np.argmin() result back to the correct untreated_match it requires that the untreated dataframe maintains a consistent order relative to the distance matrix. If the dataframe is either shuffled or filtered without resetting the index, the code may return the wrong Z values. 

In [4]:
from scipy.spatial.distance import cdist

# 1. Vectorized calculation for distances between all treated and untreated
treated_z = treated[['Z1', 'Z2']].values
untreated_z = untreated[['Z1', 'Z2']].values

# Using cdist to find distances between all pairs
all_dists = cdist(treated_z, untreated_z, metric='mahalanobis', VI=inv_cov)

# 2. Get nearest neighbor distance for each treated row
min_dists_per_treated = np.min(all_dists, axis=1)

# 3. Find the index where the "nearest neighbor distance" is maximum
outlier_idx = np.argmax(min_dists_per_treated)
max_dist = min_dists_per_treated[outlier_idx]

# 4. Extract Z values for the treated item and its closest match
treated_outlier = treated.iloc[outlier_idx]
match_idx_in_untreated = np.argmin(all_dists[outlier_idx])
untreated_match = untreated.iloc[match_idx_in_untreated]

print(f"Treated item with least common support (Index {treated_outlier.name}):")
print(f"  Z1: {treated_outlier['Z1']:.4f}, Z2: {treated_outlier['Z2']:.4f}")
print(f"  Smallest distance to control group: {max_dist:.4f}")
print(f"\nNearest untreated neighbor (Index {untreated_match.name}):")
print(f"  Z1: {untreated_match['Z1']:.4f}, Z2: {untreated_match['Z2']:.4f}")

Treated item with least common support (Index 494):
  Z1: 2.6962, Z2: 0.5382
  Smallest distance to control group: 1.3830

Nearest untreated neighbor (Index 418):
  Z1: 1.5200, Z2: -1.2822
